# ViRead VietOCR V2 — huấn luyện production sau khi Locked Test sẵn sàng

Dùng notebook này để chạy **job huấn luyện production đầy đủ**. Backend chỉ xếp job khi Train/Validation và Locked Test độc lập đạt điều kiện. Worker nhận tối đa một job READY, train trên toàn bộ Train/Validation đã đăng ký, rồi benchmark candidate bằng Locked Test đã niêm phong. Mẫu Locked Test không được đưa vào dữ liệu train. Model đang hoạt động chỉ đổi sau khi quality gate đạt và Admin phê duyệt.

Trong Colab, chọn GPU runtime và thêm Secrets `VIREAD_API_BASE`, `VIREAD_TRAINING_WORKER_TOKEN` (tùy chọn `VIREAD_REPOSITORY_REF`, mặc định `main`). Chạy **Runtime → Run all** một lần. Nếu hiện “No ready job”, chưa có training nào bắt đầu; kiểm tra trạng thái Locked Test trong ViRead Admin. Sau khi job hoàn tất, xem run, benchmark và quality gate tại Admin. Nếu còn job khác trong hàng đợi thì chạy notebook lại.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import subprocess, sys, os

def secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing Colab Secret: {name}")
    return value

API_BASE = secret("VIREAD_API_BASE").rstrip("/")
WORKER_TOKEN = secret("VIREAD_TRAINING_WORKER_TOKEN")
REPOSITORY_REF = secret("VIREAD_REPOSITORY_REF", required=False) or "main"
REPOSITORY_URL = "https://github.com/trancuong3/capstone1.git"
REPO = Path("/content/viread-capstone1")
drive.mount("/content/drive")

if not REPO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPOSITORY_REF], check=True)
dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], check=True, capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError("Worker source checkout is modified; use a clean runtime directory.")
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(REPO / "ai/vietocr_v2/requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-deps", "vietocr==0.3.12"], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
from ai.vietocr_v2.colab_worker import run_next_job_once

from ai.vietocr_v2.train import prepare_vietocr_training_runtime
runtime = prepare_vietocr_training_runtime()
print(f"Training dependencies ready: NumPy {runtime['numpy_version']}, Pillow {runtime['pillow_version']}, prefetch-generator {runtime['prefetch_generator_version']}, PyTorch {runtime['torch_version']}, torchvision {runtime['torchvision_version']}, CUDA available={runtime['cuda_available']}")

PERSISTENT_WORK = Path("/content/drive/MyDrive/ViRead/VietOCR-V2-worker")
PERSISTENT_WORK.mkdir(parents=True, exist_ok=True)
print("Checking for one ready training job. Secret values will not be displayed.")
processed = run_next_job_once(API_BASE, WORKER_TOKEN, PERSISTENT_WORK)
if processed:
    print("One training job finished processing. Check ViRead Admin for its run, benchmark, and quality-gate status.")
else:
    print("No ready job was available. No training run was started.")
